# BERT - Text Classification
This notebook demonstrates how to build a sentiment classification model using BERT on the IMDB movie review dataset.

We'll walk through data loading, preprocessing, model creation, training, evaluation, and prediction.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Import the necessary libraries
This section loads all required libraries for:
- Model building (PyTorch, Transformers)
- Dataset handling and metrics (pandas, scikit-learn)
- Tokenization and scheduling (Transformers)


In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import time

## Step 2: Import the IMDB dataset and preprocess it

### About Dataset

IMDB dataset having 50K movie reviews for natural language processing or Text analytics.
This is a dataset for binary sentiment classification containing substantially more data than previous benchmark datasets. We provide a set of 25,000 highly polar movie reviews for training and 25,000 for testing. So, predict the number of positive and negative reviews using either classification or deep learning algorithms.
For more dataset information, please go through the following link,

http://ai.stanford.edu/~amaas/data/sentiment/

The `load_imdb_data()` function:
- Accepts a CSV file containing two columns: `review` (text) and `sentiment` (label: "positive"/"negative")
- Converts sentiments into binary labels: 1 for positive, 0 for negative
- Returns: a tuple of lists (texts, labels)

In [3]:
def load_imdb_data(data_file):
    df = pd.read_csv(data_file)
    texts = df['review'].tolist()
    labels = [1 if sentiment == "positive" else 0 for sentiment in df['sentiment'].tolist()]
    return texts, labels

In [4]:
data_file = "/content/drive/MyDrive/JCAIEAH-003/Notes and Hands On/Modul 3/Day 4/IMDB Dataset.csv"
texts, labels = load_imdb_data(data_file)

In [5]:
texts[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [6]:
labels[0]

1

## Step 3: Create a custom dataset class for text classification
`TextClassificationDataset` is a subclass of PyTorch `Dataset`.
- Parameters:
    - `texts`: list of review strings
    - `labels`: list of corresponding binary labels
    - `tokenizer`: BERT tokenizer for preprocessing
    - `max_length`: maximum sequence length for BERT
- Returns a dictionary with `input_ids`, `attention_mask`, and `label` tensors per example.

In [7]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(label)}

## Step 4: Build our custom BERT classifier
`BERTClassifier` is a PyTorch neural network:
- Initializes a pre-trained BERT model
- Adds a dropout layer and a linear classification layer
- The `forward()` function:
    - Takes `input_ids` and `attention_mask` as input
    - Outputs logits for each class (2 in this case: positive or negative)

In [8]:
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            pooled_output = outputs.pooler_output
            x = self.dropout(pooled_output)
            logits = self.fc(x)
            return logits

## Step 5: Define the training function
`train()` runs one epoch of model training:
- Sets the model to training mode
- Iterates through batches from DataLoader
- Computes loss using `CrossEntropyLoss`
- Performs backpropagation and optimizer steps
- Updates learning rate via scheduler


In [9]:
def train(model, data_loader, optimizer, scheduler, device):
    model.train()
    for batch in data_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

## Step 6: Build the evaluation method
`evaluate()` measures model performance on validation data:
- Sets model to evaluation mode
- Disables gradient calculation with `torch.no_grad()`
- Collects predictions and compares them to ground-truth labels
- Returns:
  - Accuracy score
  - Classification report (precision, recall, f1-score)


In [10]:
def evaluate(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    return accuracy_score(actual_labels, predictions), classification_report(actual_labels, predictions)

## Step 7: Build the prediction method
`predict_sentiment()` is used for inference on raw text:
- Tokenizes the input
- Passes it through the model (in eval mode)
- Returns the predicted sentiment label ("positive" or "negative")

In [11]:
def predict_sentiment(text, model, tokenizer, device, max_length=128):
    model.eval()
    encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)
    return "positive" if preds.item() == 1 else "negative"

## Step 8: Define model parameters
Sets up various hyperparameters including:
- `bert_model_name`: pre-trained BERT model to use
- `num_classes`: number of sentiment classes (2)
- `max_length`: max token length per input
- `batch_size`, `num_epochs`, `learning_rate`

In [12]:
# Set up parameters
bert_model_name = 'bert-base-uncased'
num_classes = 2
max_length = 128
batch_size = 128
num_epochs = 4
learning_rate = 2e-5

## Step 9: Split the data
- Uses `train_test_split()` from scikit-learn
- Splits texts and labels into training and validation sets (80/20)


In [13]:
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

In [16]:
len(train_texts)

40000

In [19]:
len(val_texts)

10000

## Step 10: Initialize tokenizer, dataset, and data loader
- Loads BERT tokenizer
- Converts text data into PyTorch datasets
- Wraps datasets into DataLoaders for batch processing

In [20]:
tokenizer = BertTokenizer.from_pretrained(bert_model_name)
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_length)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Step 11: Set up the device and model
- Chooses GPU if available, otherwise CPU
- Loads the BERT classifier onto the selected device

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier(bert_model_name, num_classes).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step 12: Set up optimizer and learning rate scheduler
- Uses `AdamW` optimizer (recommended for Transformers)
- Defines a linear warm-up schedule with no warm-up steps

In [22]:
optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

## Step 13: Train the model
- Runs multiple epochs of training and validation
- Prints accuracy and classification report after each epoch
- Saves the trained model's state to disk using `torch.save()`

In [ ]:
for epoch in range(num_epochs):
    start_time = time.time()
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train(model, train_dataloader, optimizer, scheduler, device)
    accuracy, report = evaluate(model, val_dataloader, device)
    end_time = time.time()
    epoch_time = end_time - start_time
    print(f"Validation Accuracy: {accuracy:.4f}")
    print(report)
    print(f"Epoch {epoch + 1} took {epoch_time:.2f} seconds")

Epoch 1/4
Validation Accuracy: 0.8853
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

Epoch 1 took 885.42 seconds
Epoch 2/4
Validation Accuracy: 0.8905
              precision    recall  f1-score   support

           0       0.89      0.88      0.89      4961
           1       0.89      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

Epoch 2 took 886.91 seconds
Epoch 3/4
Validation Accuracy: 0.8934
              precision    recall  f1-score   support

           0       0.89      0.89      0.89      4961
           1       0.89      0.90      0.89      5039

    accuracy

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/JCAIEAH-003/Notes and Hands On/Modul 3/Day 4/bert_classifier.pth")

## Step 14: Evaluate the model’s predictions on new examples
- Tests the `predict_sentiment()` function with two sample texts
- Outputs the model's predicted sentiment for each

In [ ]:
# Test sentiment prediction
test_text = "The movie was great and I really enjoyed the performances of the actors."
sentiment = predict_sentiment(test_text, model, tokenizer, device)
print("The movie was great and I really enjoyed the performances of the actors.")
print(f"Predicted sentiment: {sentiment}")

The movie was great and I really enjoyed the performances of the actors.
Predicted sentiment: positive


In [ ]:
# Test sentiment prediction
test_text = "The movie was so bad and I would not recommend it to anyone."
sentiment = predict_sentiment(test_text, model, tokenizer, device)
print("The movie was so bad and I would not recommend it to anyone.")
print(f"Predicted sentiment: {sentiment}")

The movie was so bad and I would not recommend it to anyone.
Predicted sentiment: negative


## 🧾 Conclusion
In this hands-on project, we built a BERT-based sentiment classifier for movie reviews using the IMDB dataset.
### ✅ Key Results:
- The model reached a **validation accuracy of ~89%** across 4 epochs.
- The **F1-scores** for both classes (positive and negative) were consistent at around **0.89–0.90**, indicating a balanced and reliable performance.

### 📌 Insights:
- The model shows **strong generalization ability** on the validation set, with stable precision and recall across all epochs.
- It correctly predicted sentiment on sample test cases:
  - `"The movie was great and I really enjoyed the performances of the actors."` → **positive**
  - `"The movie was so bad and I would not recommend it to anyone."` → **negative**

### 🧠 What problem was solved?
We addressed the problem of **binary sentiment classification** (positive vs negative) for text data.
- Traditional ML models often struggle to capture long-range dependencies and contextual meaning in language.
- By using **pre-trained BERT**, we leveraged deep semantic understanding, improving classification performance without training from scratch.

### 🚀 Next Steps (Optional Ideas):
- Test on more custom movie reviews to further validate generalization.
- Fine-tune on domain-specific datasets (e.g., product reviews, tweets).